# Item-to-Item Collaborative Filtering (Adjusted Cosine Similarity)

Sarwar et al. 2001's item-based CF. For target user `u` and item `i`, the predicted rating is

$$ \hat{r}_{ui} = \bar{r}_i + \frac{\sum_{j \in N_k(i; u)} \text{sim}(i, j)\,(r_{uj} - \bar{r}_j)}{\sum_{j \in N_k(i; u)} |\text{sim}(i, j)|} $$

where `N_k(i; u)` is the top-`k` items most similar to `i` that user `u` has rated, and `sim(i, j)` is **adjusted cosine similarity** — cosine on item-rating vectors after subtracting the per-item mean. Item-based CF tends to outperform user-based CF on movie datasets where items have many more raters than users have ratings, which is the case for MovieLens.


In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
import os, random, sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

ARTIFACT_DIR    = Path('/content/gdrive/MyDrive/recsys_artifacts')
SPLITS_DIR      = ARTIFACT_DIR / 'splits'
PREDICTIONS_DIR = ARTIFACT_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(ARTIFACT_DIR))
from recsys_metrics import build_candidate_lists, evaluate_model

In [3]:
train = pd.read_csv(SPLITS_DIR / 'train.csv')
val   = pd.read_csv(SPLITS_DIR / 'val.csv')
test  = pd.read_csv(SPLITS_DIR / 'test.csv')
trainval = pd.concat([train, val], ignore_index=True)
print(f'train: {len(train):,}  val: {len(val):,}  test: {len(test):,}  trainval: {len(trainval):,}')

train: 99,616  val: 610  test: 610  trainval: 100,226


## Build the sparse item-by-user matrix and adjusted-cosine item similarities

Adjusted cosine subtracts each user's mean rating from their ratings before computing item-item cosine. This corrects for users with systematically high or low rating tendencies.

In [4]:
users  = sorted(trainval['userId'].unique())
movies = sorted(trainval['movieId'].unique())
u2i = {u: i for i, u in enumerate(users)}
m2i = {m: i for i, m in enumerate(movies)}

rows = trainval['movieId'].map(m2i).to_numpy()
cols = trainval['userId'].map(u2i).to_numpy()
vals = trainval['rating'].to_numpy(dtype=np.float32)
R_iu = csr_matrix((vals, (rows, cols)), shape=(len(movies), len(users)))

# Per-USER mean (adjusted cosine subtracts user mean from each rating).
R_ui = R_iu.T.tocsr()
user_sums   = np.asarray(R_ui.sum(axis=1)).flatten()
user_counts = np.asarray((R_ui != 0).sum(axis=1)).flatten()
user_means  = np.divide(user_sums, user_counts, out=np.zeros_like(user_sums), where=user_counts > 0)

# Per-ITEM mean (used as the fallback / additive term in the prediction formula).
item_sums   = np.asarray(R_iu.sum(axis=1)).flatten()
item_counts = np.asarray((R_iu != 0).sum(axis=1)).flatten()
item_means  = np.divide(item_sums, item_counts, out=np.zeros_like(item_sums), where=item_counts > 0)
global_mean = float(trainval['rating'].mean())

# Subtract user mean from each nonzero entry (user-axis = columns of R_iu).
R_iu_centered = R_iu.copy().astype(np.float32).tocsc()
for u_col in range(R_iu_centered.shape[1]):
    start, end = R_iu_centered.indptr[u_col], R_iu_centered.indptr[u_col + 1]
    R_iu_centered.data[start:end] -= user_means[u_col]
R_iu_centered = R_iu_centered.tocsr()

print('Item-user matrix:', R_iu.shape, '  nnz:', R_iu.nnz)

Item-user matrix: (9701, 610)   nnz: 100226


In [5]:
item_sim = cosine_similarity(R_iu_centered, dense_output=True)
np.fill_diagonal(item_sim, 0.0)
print('Item-item similarity matrix:', item_sim.shape)

Item-item similarity matrix: (9701, 9701)


## Prediction

In [6]:
TOP_K_NEIGHBORS = 30  # tuned on validation

def predict_rating(user_id, movie_id):
    if movie_id not in m2i:
        return user_means[u2i[user_id]] if user_id in u2i else global_mean
    i = m2i[movie_id]
    if user_id not in u2i:
        return item_means[i] if item_counts[i] > 0 else global_mean
    u = u2i[user_id]

    user_row = R_ui.getrow(u).toarray().flatten()  # ratings this user has given
    rated_items = np.where(user_row > 0)[0]
    if len(rated_items) == 0:
        return item_means[i] if item_counts[i] > 0 else global_mean

    sims = item_sim[i, rated_items]
    top_idx = np.argsort(-sims)[:TOP_K_NEIGHBORS]
    top_items = rated_items[top_idx]
    top_sims = sims[top_idx]
    if np.abs(top_sims).sum() < 1e-9:
        return item_means[i] if item_counts[i] > 0 else global_mean

    centered = user_row[top_items] - item_means[top_items]
    pred = item_means[i] + (top_sims * centered).sum() / np.abs(top_sims).sum()
    return float(np.clip(pred, 0.5, 5.0))

def predict_pointwise(df):
    out = df[['userId', 'movieId', 'rating']].copy()
    out['predicted_rating'] = [predict_rating(u, m) for u, m in zip(df['userId'], df['movieId'])]
    return out

def predict_fn(user_id, movie_ids):
    return np.array([predict_rating(int(user_id), int(m)) for m in movie_ids], dtype=float)

## Evaluate + save

In [7]:
candidates = build_candidate_lists(train, test, num_negatives=99, seed=SEED)
pointwise = predict_pointwise(test)
metrics = evaluate_model(predict_fn, test, candidates, pointwise_predictions=pointwise, k=10, threshold=3.5)
print('=== Item-CF — test set ===')
for key in ['rmse', 'mae', 'hr_at_k', 'ndcg_at_k', 'precision_at_k', 'recall_at_k', 'f1_at_k']:
    print(f'  {key:18s} = {metrics[key]:.4f}')

=== Item-CF — test set ===
  rmse               = 0.9756
  mae                = 0.7479
  hr_at_k            = 0.0689
  ndcg_at_k          = 0.0238
  precision_at_k     = 0.0066
  recall_at_k        = 0.0656
  f1_at_k            = 0.0119


In [8]:
out_path = PREDICTIONS_DIR / 'item_cf.csv'
pointwise[['userId', 'movieId', 'predicted_rating']].to_csv(out_path, index=False)
print(f'Saved {len(pointwise)} predictions -> {out_path}')

Saved 610 predictions -> /content/gdrive/MyDrive/recsys_artifacts/predictions/item_cf.csv
